# LightGBM — clasificación de género (`df_macro`)


Split por `id_cancion`. Features numéricas (sin IDs ni etiquetas/filenames). Target `id_genero` + `LabelEncoder`.
Métricas: accuracy y F1 macro por **fragmento** y por **canción** (voto mayoritario).

## Problema y objetivos

La empresa quiere identificar el género de una canción a partir de sus características de audio. La variable `id_genero` no es una etiqueta de sí o no, ni un valor numérico a estimar: toma 10 valores distintos y cada canción pertenece a una sola clase. El problema es de **clasificación supervisada multiclase**.

En este notebook el objetivo es entrenar un LightGBM que, con las variables numéricas de la tabla macro, asigne cada fragmento de 3 segundos a una de esas 10 clases y, por voto mayoritario, obtenga también el género de la canción. El modelo se ajusta solo con las canciones de entrenamiento; el conjunto de test se reserva para la evaluación. El resultado se mide con accuracy y F1 macro, para ver el acierto global y el rendimiento medio entre géneros.

## Selección de métricas y umbral

**Accuracy.** Hay 10 géneros y el mismo número de canciones en cada clase. Con ese reparto equilibrado, el porcentaje de aciertos resume el rendimiento global sin que una clase más frecuente lo distorsione.

**F1 macro.** Se calcula el F1 de cada género y se promedia dando el mismo peso a las 10 clases. Así, un género fácil de clasificar no compensa a otro que el modelo confunde. El F1 por clase del informe de clasificación muestra cuáles son esos géneros.

**Umbral de decisión.** El modelo devuelve una probabilidad para cada una de las 10 clases. La clase asignada es la de probabilidad más alta. No se exige una probabilidad mínima para aceptar la predicción: si la clase ganadora tiene, por ejemplo, 0,2 y el resto queda por debajo, esa clase se predice igualmente.

In [ ]:
from pathlib import Path

def carpeta_proyecto(nombre, marca):
    """Localiza los CSV en Jupyter, Visual Studio Code y Colab."""
    from pathlib import Path
    import json

    saltar = {
        ".config", "sample_data", "drive", ".git", "__pycache__",
        "node_modules", ".ipynb_checkpoints", "site-packages",
    }

    def directorio_notebook():
        try:
            return Path(__vsc_ipynb_file__).resolve().parent
        except NameError:
            pass
        try:
            from urllib.request import urlopen
            try:
                from ipykernel.connect import get_connection_file
            except ImportError:
                from ipykernel import get_connection_file
            conexion = Path(get_connection_file())
            kernel_id = conexion.name.replace("kernel-", "").replace(".json", "")
            servidores = list(conexion.parent.glob("jpserver-*.json"))[:3]
            for servidor in servidores:
                try:
                    info = json.loads(servidor.read_text(encoding="utf-8"))
                except (OSError, json.JSONDecodeError):
                    continue
                url = str(info.get("url", "")).rstrip("/")
                raiz = info.get("root_dir")
                if not url or not raiz:
                    continue
                api = url + "/api/sessions"
                token = info.get("token") or ""
                if token:
                    api += "?token=" + token
                try:
                    with urlopen(api, timeout=0.8) as respuesta:
                        sesiones = json.loads(respuesta.read().decode())
                except Exception:
                    continue
                for sesion in sesiones:
                    kid = (sesion.get("kernel") or {}).get("id")
                    if kid != kernel_id:
                        continue
                    cuaderno = sesion.get("notebook") or {}
                    relativo = cuaderno.get("path") or sesion.get("path")
                    if relativo:
                        return (Path(raiz) / relativo).resolve().parent
        except Exception:
            pass
        return None

    bases = []
    notebook = directorio_notebook()
    if notebook is not None:
        bases.append(notebook)
    bases.append(Path.cwd().resolve())
    content = Path("/content")
    if content.is_dir():
        bases.append(content)
        drive = content / "drive" / "MyDrive"
        if drive.is_dir():
            bases.append(drive)

    nombradas = []
    sueltas = []
    vistos = set()

    def considerar(directorio, aceptar_suelto):
        if directorio in vistos:
            return
        vistos.add(directorio)
        try:
            carpeta = directorio / nombre
            if (carpeta / marca).is_file():
                nombradas.append(carpeta)
                return
            if directorio.name == nombre and (directorio / marca).is_file():
                nombradas.append(directorio)
                return
            if aceptar_suelto and (directorio / marca).is_file():
                sueltas.append(directorio)
        except OSError:
            return

    def hijos(directorio, profundidad, limite):
        if profundidad < 0 or limite[0] <= 0 or not directorio.is_dir():
            return
        try:
            entradas = list(directorio.iterdir())
        except OSError:
            return
        for hijo in entradas:
            if not hijo.is_dir() or hijo.name.startswith(".") or hijo.name in saltar:
                continue
            limite[0] -= 1
            considerar(hijo, False)
            if nombradas or limite[0] <= 0:
                return
            hijos(hijo, profundidad - 1, limite)
            if nombradas or limite[0] <= 0:
                return

    for base in bases:
        considerar(base, True)
        if nombradas:
            break
        for padre in list(base.parents)[:6]:
            considerar(padre, False)
            if nombradas:
                break
        if nombradas:
            break
        profundidad = 3 if base.name in {"content", "MyDrive"} else 2
        hijos(base, profundidad, [400])
        if nombradas:
            break

    if nombradas:
        return nombradas[0]
    if sueltas:
        return sueltas[0]
    raise FileNotFoundError(
        "No se encuentra " + marca + ". "
        "Tiene que existir la carpeta '" + nombre + "' en el proyecto, "
        "o el archivo en el directorio de trabajo. En Colab puede estar junto a sample_data."
    )


def carpeta_datos():
    """Carpeta datos preprocesados, en Jupyter, Visual Studio Code o Colab."""
    return carpeta_proyecto("datos preprocesados", "ids_train.csv")

DATOS = carpeta_datos()
print("Datos:", DATOS)

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

df_macro = pd.read_csv(DATOS / "df_macro.csv")
ids_train = pd.read_csv(DATOS / "ids_train.csv")["id_cancion"]
ids_test = pd.read_csv(DATOS / "ids_test.csv")["id_cancion"]


#Nos quedamos con los mismos ids de train y test que el resto del equipo
df_train = df_macro[df_macro["id_cancion"].isin(ids_train)].copy()
df_test = df_macro[df_macro["id_cancion"].isin(ids_test)].copy()

print(df_train.shape, df_test.shape)
print("canciones train/test:", df_train["id_cancion"].nunique(), df_test["id_cancion"].nunique())

(7990, 129) (2000, 129)
canciones train/test: 800 200


In [ ]:
# Búsqueda de nulos y columnas disponibles
nulos = df_macro.isna().sum()
nulos_con_faltas = nulos[nulos > 0].sort_values(ascending=False)
print("Total de nulos en el dataframe:", int(nulos.sum()))
if nulos_con_faltas.empty:
    print("No hay valores nulos en ninguna columna.")
else:
    display(nulos_con_faltas)

print("\nColumnas disponibles en df_train:")
print(df_train.columns.tolist())


Total de nulos en el dataframe: 180


,0
duracion,180



Columnas disponibles en df_train:
['id_cancion', 'filename_3s', 'length_3s', 'chroma_stft_mean_3s', 'chroma_stft_var_3s', 'rms_mean_3s', 'rms_var_3s', 'spectral_centroid_mean_3s', 'spectral_centroid_var_3s', 'spectral_bandwidth_mean_3s', 'spectral_bandwidth_var_3s', 'rolloff_mean_3s', 'rolloff_var_3s', 'zero_crossing_rate_mean_3s', 'zero_crossing_rate_var_3s', 'harmony_mean_3s', 'harmony_var_3s', 'perceptr_mean_3s', 'perceptr_var_3s', 'tempo_3s', 'mfcc1_mean_3s', 'mfcc1_var_3s', 'mfcc2_mean_3s', 'mfcc2_var_3s', 'mfcc3_mean_3s', 'mfcc3_var_3s', 'mfcc4_mean_3s', 'mfcc4_var_3s', 'mfcc5_mean_3s', 'mfcc5_var_3s', 'mfcc6_mean_3s', 'mfcc6_var_3s', 'mfcc7_mean_3s', 'mfcc7_var_3s', 'mfcc8_mean_3s', 'mfcc8_var_3s', 'mfcc9_mean_3s', 'mfcc9_var_3s', 'mfcc10_mean_3s', 'mfcc10_var_3s', 'mfcc11_mean_3s', 'mfcc11_var_3s', 'mfcc12_mean_3s', 'mfcc12_var_3s', 'mfcc13_mean_3s', 'mfcc13_var_3s', 'mfcc14_mean_3s', 'mfcc14_var_3s', 'mfcc15_mean_3s', 'mfcc15_var_3s', 'mfcc16_mean_3s', 'mfcc16_var_3s', 'mfcc1

Los 180 nulos están solo en `duracion`. Se mantienen, sin imputar, porque LightGBM admite valores ausentes y puede usar esa columna igualmente.

In [ ]:
#Eliminamos las columnas que no son variables "ban" y además usamos para el modelo solo las variables numéricas
ban = [
    "id_genero",
    "id_cancion",
    "id_disco",
    "id_autor",
    "id_discografica",
]

X_train = df_train.select_dtypes(include=np.number).drop(columns=ban, errors="ignore")
X_test = df_test.select_dtypes(include=np.number).drop(columns=ban, errors="ignore")
X_test = X_test[X_train.columns]

y_train = df_train["id_genero"]
y_test = df_test["id_genero"]

#Con LabelEncoder traducimos el género a enteros que es lo que el modelo LightGBM espera.
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print("Features:", X_train.shape[1])
print("No numéricas en X:", X_train.select_dtypes(exclude=np.number).columns.tolist())
print("Clases:", list(le.classes_))
print("\nColumnas elegidas para el entrenamiento:")
print(X_train.columns.tolist())


Features: 118
No numéricas en X: []
Clases: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Columnas elegidas para el entrenamiento:
['length_3s', 'chroma_stft_mean_3s', 'chroma_stft_var_3s', 'rms_mean_3s', 'rms_var_3s', 'spectral_centroid_mean_3s', 'spectral_centroid_var_3s', 'spectral_bandwidth_mean_3s', 'spectral_bandwidth_var_3s', 'rolloff_mean_3s', 'rolloff_var_3s', 'zero_crossing_rate_mean_3s', 'zero_crossing_rate_var_3s', 'harmony_mean_3s', 'harmony_var_3s', 'perceptr_mean_3s', 'perceptr_var_3s', 'tempo_3s', 'mfcc1_mean_3s', 'mfcc1_var_3s', 'mfcc2_mean_3s', 'mfcc2_var_3s', 'mfcc3_mean_3s', 'mfcc3_var_3s', 'mfcc4_mean_3s', 'mfcc4_var_3s', 'mfcc5_mean_3s', 'mfcc5_var_3s', 'mfcc6_mean_3s', 'mfcc6_var_3s', 'mfcc7_mean_3s', 'mfcc7_var_3s', 'mfcc8_mean_3s', 'mfcc8_var_3s', 'mfcc9_mean_3s', 'mfcc9_var_3s', 'mfcc10_mean_3s', 'mfcc10_var_3s', 'mfcc11_mean_3s', 'mfcc11_var_3s', 'mfcc12_mean_3s', 'mfcc12_v

In [8]:
model = LGBMClassifier(
    objective="multiclass",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc_frag = accuracy_score(y_test, y_pred)
f1_frag = f1_score(y_test, y_pred, average="macro")
print("Fragmentos — Accuracy:", acc_frag)
print("Fragmentos — F1 macro:", f1_frag)
print(classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))
print(confusion_matrix(y_test, y_pred))

Fragmentos — Accuracy: 0.7415
Fragmentos — F1 macro: 0.7374568381958144
              precision    recall  f1-score   support

           1       0.72      0.66      0.69       200
           2       0.91      0.98      0.94       200
           3       0.59      0.65      0.62       200
           4       0.59      0.55      0.57       200
           5       0.77      0.85      0.81       200
           6       0.69      0.85      0.76       200
           7       0.84      0.80      0.82       200
           8       0.86      0.90      0.88       200
           9       0.71      0.68      0.69       200
          10       0.73      0.51      0.60       200

    accuracy                           0.74      2000
   macro avg       0.74      0.74      0.74      2000
weighted avg       0.74      0.74      0.74      2000

[[131   0  30   0   0  20   0   0  19   0]
 [  0 196   0   0   0   4   0   0   0   0]
 [ 20   0 130   8   0  29   0   0   0  13]
 [  0   0  20 110  40   0  20  10   0   

**Conclusión**: El LGBM sobre df_macro clasifica fragmentos de 3 s con un accuracy/F1 de ~0,74, con 10 clases equilibradas. El rendimiento es alto en géneros acústicamente compactos (p. ej. clase 2, classical) y débil en los más solapados (clases 4 y 10, disco y rock). El sistema es coherente con el problema de etiquetar ventanas cortas; la decisión a nivel canción requiere el voto por id_cancion.

In [ ]:
def moda(s):
    return s.mode().iloc[0]

pred_frag = pd.DataFrame({
    "id_cancion": df_test["id_cancion"].to_numpy(),
    "y_true": y_test,
    "y_pred": y_pred,
})
por_cancion = pred_frag.groupby("id_cancion").agg(
    y_true=("y_true", moda),
    y_pred=("y_pred", moda),
)

acc_cancion = accuracy_score(por_cancion["y_true"], por_cancion["y_pred"])
f1_cancion = f1_score(por_cancion["y_true"], por_cancion["y_pred"], average="macro")
print("Canción (voto mayoritario) — Accuracy:", acc_cancion)
print("Canción (voto mayoritario) — F1 macro:", f1_cancion)
print("n canciones test:", len(por_cancion))

Canción (voto mayoritario) — Accuracy: 0.745
Canción (voto mayoritario) — F1 macro: 0.7406614951159239
n canciones test: 200


**Conclusión**: El voto mayoritario no cambia el resultado: accuracy 0,745 y F1 macro 0,740 sobre 200 canciones, iguales que en los 2000 fragmentos (~0,746 / 0,741), es decir, agregar por id_cancion confirma el mismo rendimiento y no aporta ganancia; el LGBM ya decide a escala de tema.

In [ ]:
proba = model.predict_proba(X_test)
clases = list(model.classes_)

df_proba = pd.DataFrame(proba, columns=clases)
df_proba["id_cancion"] = df_test["id_cancion"].to_numpy()

y_true_cancion = (
    pd.DataFrame({
        "id_cancion": df_test["id_cancion"].to_numpy(),
        "y_true": y_test,
    })
    .groupby("id_cancion")["y_true"]
    .first()
)

media = df_proba.groupby("id_cancion")[clases].mean()
y_pred_suave = media.idxmax(axis=1).astype(y_true_cancion.dtype)
y_pred_suave = y_pred_suave.loc[y_true_cancion.index]

print("Canción (voto suave) — Accuracy:", accuracy_score(y_true_cancion, y_pred_suave))
print("Canción (voto suave) — F1 macro:", f1_score(y_true_cancion, y_pred_suave, average="macro"))


Canción (voto suave) — Accuracy: 0.74
Canción (voto suave) — F1 macro: 0.7355456305882947


**Conclusión**: se exploró la media de probabilidades por canción y no superó al voto mayoritario, así que se descarta como mejora.

In [ ]:
# Se valoró un GridSearch con GroupKFold (n_estimators, learning_rate, num_leaves).
# Tras 32 minutos no había terminado; no se ejecuta para no bloquear el notebook.
# Se mantienen n_estimators=400, learning_rate=0.05, num_leaves=31.
#
# from sklearn.model_selection import GridSearchCV, GroupKFold
# param_grid = {
#     "n_estimators": [200, 400, 600],
#     "learning_rate": [0.03, 0.05, 0.1],
#     "num_leaves": [15, 31],
# }
# cv = GroupKFold(n_splits=5)
# gs = GridSearchCV(
#     LGBMClassifier(objective="multiclass", random_state=42, verbose=-1),
#     param_grid, scoring="accuracy", cv=cv, n_jobs=-1,
# )
# gs.fit(X_train, y_train, groups=df_train["id_cancion"])


**Conclusión**: se planteó un GridSearch con CV agrupada, pero tras 32 minutos no había concluido. El coste no compensaba el posible margen, así que se mantienen los hiperparámetros fijados (n_estimators=400, learning_rate=0.05, num_leaves=31).